# Site Assessment Data (SAD)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import brodata

In [ ]:
sad = brodata.sad.SiteAssessmentData.from_bro_id("SAD000000011742")

In [ ]:
sad.to_dict()

In [ ]:
# plot the geometry of this Site Assessment Data, together with the measurementPoints
fig, ax = plt.subplots(figsize=(8, 8))
gpd.GeoDataFrame(geometry=[sad.geometry]).plot(ax=ax)
sad.measurementPoint.plot(ax=ax, color="red", markersize=50)
for index in sad.measurementPoint.index:
    point = sad.measurementPoint.geometry.loc[index]
    ax.annotate(index, (point.x, point.y), textcoords="offset points", xytext=(0,10), ha='center')
ax.axis("equal");

In [ ]:
# show the contents of `measurementPoint`
sad.measurementPoint

In [ ]:
# Plot lithology logs for all measurement points
f, ax = plt.subplots(figsize=(15,6))
for i, name in enumerate(sad.measurementPoint.index):
    df = sad.measurementPoint.at[name,'DescriptiveBoreholeLog']['layer']
    brodata.plot.bro_lithology_advanced(df, x=i, width=0.6, soil_name_column='soilName', ax=ax, bro_id=name)
    # plot filter if available
    if isinstance(sad.measurementPoint.at[name,'filter'], pd.DataFrame):
        fs = sad.measurementPoint.at[name,'filter']
        for fi in fs.index:
            y = [ -fs.at[fi,'upperBoundary'], -fs.at[fi,'lowerBoundary'] ]
            ax.plot([i, i], y, color='k', linewidth=3, linestyle=':', solid_capstyle="butt")
ax.set_xlim(-0.5, len(sad.measurementPoint) - 0.5)
ax.set_xticks(range(len(sad.measurementPoint)))
ax.set_xticklabels(sad.measurementPoint.index, fontdict={'rotation':45, 'ha':'right'})
ax.set_ylim(-sad.measurementPoint['finalDepth'].max()-0.1, 0.0)
ax.set_axisbelow(True)
ax.grid(True)
ax.set_ylabel('Depth (m)');

In [ ]:
# the samling analysis for a specific measurement point is hidden deep in the data structure (this will change in future versions of `brodata)
df = sad.measurementPoint.loc["1786310", "filter"]['groundwaterSampling'].iloc[0]['groundwaterSampleAnalysis'].iloc[0]['analysis'].iloc[0]

# add the parameter description
parameter_list = brodata.gar.get_parameter_list()
# add a description when parameter is in parameter list
df["parameter_description"] = ""
for index in df.index:
    param = df.at[index, "parameter"]
    if param in parameter_list.index:
        df.at[index, "parameter_description"] = parameter_list.at[param, "description"]
df